In [0]:
from pathlib import Path
import sys
import os
import json
from datetime import datetime, timezone
from destination import DEFAULT_INGEST_CATEGORIES

REPO_ROOT = Path(
    "/Workspace/Users/tuvu.uwyo@gmail.com/weather_intelligence_databricks"
)
sys.path.insert(0, str(REPO_ROOT))

from destination_api import DestinationAPI, DestinationAPIError

In [0]:
CATALOG = "weather_intelligence"
SCHEMA = "bronze"
TABLE = "destinations_raw"

In [0]:
# --- 1. Set up the catalog, schema, and Bronze table (safe to re-run) ---
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.{TABLE} (
        place_id STRING,
        location STRING NOT NULL,
        raw_feature STRING NOT NULL,
        fetched_at TIMESTAMP NOT NULL,
        source STRING NOT NULL
    ) USING DELTA
""")

In [0]:
# --- 2. Fetch raw features from Geoapify — same call destination.py already makes ---

os.environ["GEOAPIFY_API_KEY"] = '649d5b3a92ed4bbfa97f8c40906640f4'
LOCATION = "Honolulu, HI"  # change if you want a different test location

client = DestinationAPI(api_key=os.environ["GEOAPIFY_API_KEY"])
features = client.get_places_radius(LOCATION, categories=DEFAULT_INGEST_CATEGORIES)

print(f"Fetched {len(features)} raw features for {LOCATION}")

In [0]:
json.dumps(features[0])

In [0]:
features[0].get("properties", {}).get("place_id")

In [0]:
# --- 3. Build rows and write them to Bronze, untouched ---
now = datetime.now(timezone.utc)

rows = [
    {
        "place_id": feature.get("properties", {}).get("place_id")
        ,"location": LOCATION
        ,"raw_feature": json.dumps(feature)
        ,"fetched_at": now
        ,"source": "geoapify"
    }
    for feature in features
]

df = spark.createDataFrame(rows)
df.write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")

print(f"Wrote {df.count()} rows to {CATALOG}.{SCHEMA}.{TABLE}")

In [0]:
# --- 4. check the Spark table ---
display(spark.read.table(f"{CATALOG}.{SCHEMA}.{TABLE}"))

In [0]:
# Or, SQL directly
spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.{TABLE} LIMIT 5").show(truncate=False)

In [0]:
row = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE}").limit(1).collect()[0]
feature = json.loads(row["raw_feature"])
# print(feature["properties"]["name"])
print(feature["properties"]["place_id"])

In [0]:
feature["properties"].keys()

In [0]:
existing_columns = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE}").columns
existing_columns

In [0]:
# --- 5. Add new COLUMN raw_detail to the Bronze table ---
if "raw_detail" not in existing_columns:
    spark.sql(f"""
        ALTER TABLE {CATALOG}.{SCHEMA}.{TABLE}
        ADD COLUMNS (raw_detail STRING)
    """)
    print("Added raw_detail column")
else:
    print("raw_detail column already exists")

In [0]:
# --- 5.2. Find which rows still need a detail fetch ---
rows_needing_detail = (
    spark.table(f"{CATALOG}.{SCHEMA}.{TABLE}")
    .where(f"location = '{LOCATION}' AND raw_detail IS NULL")
    .select("place_id")
    .collect()
)
rows_needing_detail

In [0]:
len(rows_needing_detail)

In [0]:
place_ids = [row["place_id"] for row in rows_needing_detail if row["place_id"] is not None]
print(f"Fetching details for {len(place_ids)} places — this makes one API call per place, so expect it to take a bit")


In [0]:
# --- 5.3. Fetch get_place_details() for each — one call per place ---
detail_rows = []

for place_id in place_ids:
    try:
        detail = client.get_place_details(place_id)
        detail_rows.append({
            "place_id": place_id
            ,"raw_detail": json.dumps(detail)
        })
    except DestinationAPIError as e:
        print(f"Skipped {place_id}: {e}")

print(f"Fetched {len(detail_rows)} details ({len(place_ids) - len(detail_rows)} skipped)")


In [0]:
# --- 5.4. Merge the fetched details back into Bronze, matched by place_id ---
if detail_rows:
    detail_df = spark.createDataFrame(detail_rows)
    detail_df.createOrReplaceTempView("detail_updates")

    spark.sql(f"""
        MERGE INTO {CATALOG}.{SCHEMA}.{TABLE} AS target
        USING detail_updates AS source
        ON target.place_id = source.place_id
        WHEN MATCHED THEN UPDATE SET target.raw_detail = source.raw_detail
    """)

    print("Merged raw_detail into Bronze")
else:
    print("Nothing to merge")

In [0]:
# --- 5.5. Confirm one row round-tripped correctly ---
sample = (
    spark.table(f"{CATALOG}.{SCHEMA}.{TABLE}")
    .where("raw_detail IS NOT NULL")
    .limit(1)
    .collect()[0]
)
detail = json.loads(sample["raw_detail"])
print(detail["properties"].get("name"))

In [0]:
spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.{TABLE} LIMIT 3").show(truncate=False)